# Predição de Conversão de Leads

Modelo para prever probabilidade de conversão de leads.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

# modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# carregar base
df = pd.read_excel('../cadastros_08-052026 22_42_48.xls')

df.head()

## Definição do Target

Target atual: `converteu`

In [ ]:
# =========================
# TARGET
# =========================

# AJUSTAR DE ACORDO COM A BASE REAL

# exemplos possíveis:
# df['converteu'] = ...


# Feature Engineering

Ajustar conforme disponibilidade da base.


In [ ]:
# exemplo de features
possible_features = [
    'Cidade',
    'Perfil',
    'Origem',
    'Corretor',
    'Tipo Imóvel',
    'Faixa Valor',
]

features = [f for f in possible_features if f in df.columns]

print(features)


In [ ]:
# remover linhas sem target
df_model = df.dropna(subset=['converteu'])

X = df_model[features]
y = df_model['converteu']

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_features
        ),
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median'))
            ]),
            numeric_features
        )
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)


## Modelo 1 — Regressão Logística

In [ ]:
logistic_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

preds_logistic = logistic_model.predict(X_test)

print('Accuracy:')
print(accuracy_score(y_test, preds_logistic))

print('\nClassification Report:')
print(classification_report(y_test, preds_logistic))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, preds_logistic)

plt.title('Matriz de Confusão - Regressão Logística')
plt.show()


## Modelo 2 — Random Forest

In [ ]:
rf_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

preds_rf = rf_model.predict(X_test)

print('Accuracy:')
print(accuracy_score(y_test, preds_rf))

print('\nClassification Report:')
print(classification_report(y_test, preds_rf))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, preds_rf)

plt.title('Matriz de Confusão - Random Forest')
plt.show()


# Próximos passos

- Melhorar qualidade dos dados
- Criar novas features
- Testar XGBoost
- Automatizar pipeline ETL
- Integrar modelos ao dashboard
- Criar monitoramento dos modelos
